# Task 6 — Modeling Preprocessing Pipeline

This notebook documents and validates the reusable preprocessing pipeline used **after** Tasks 1–5 define the target, final features, and Train/Validation/Test split.

Task 6 requirements covered here:

1. Missing-value handling is explicit.
2. Numeric features are prepared with median imputation.
3. Categorical features are imputed and one-hot encoded.
4. Numeric scaling is optional for scale-sensitive algorithms.
5. All learned preprocessing statistics are fit on **Train only**.
6. `Pipeline` and `ColumnTransformer` are used.
7. Validation and Test are **transform-only**.

> This task intentionally does not choose the target or split rows; those decisions belong to Tasks 1–5.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd

from scripts.modeling_preprocessing import (
    build_preprocessor,
    fit_transform_feature_splits,
    preprocessing_summary,
)


## Input contract

In the real modeling workflow, this task expects the outputs of Task 5:

```python
X_train
X_validation
X_test
```

All three must contain the same feature columns in the same order. The target (`y_train`, `y_validation`, `y_test`) is not transformed here.

## Small validation example

The following tiny dataset is **only a pipeline sanity check**. It contains representative pandas dtypes from the cleaned Divar dataset, including nullable integers, strings, booleans, missing values, and an unseen validation category.

In [ ]:
X_train_demo = pd.DataFrame({
    "building_size": pd.Series([80, 100, np.nan, 120], dtype="float64"),
    "rooms_count": pd.Series([1, 2, 2, pd.NA], dtype="Int64"),
    "city_slug": pd.Series(["tehran", "tehran", pd.NA, "karaj"], dtype="string"),
    "has_elevator": pd.Series([True, False, pd.NA, True], dtype="boolean"),
})

X_validation_demo = pd.DataFrame({
    "building_size": [90.0],
    "rooms_count": pd.Series([3], dtype="Int64"),
    "city_slug": pd.Series(["new-city"], dtype="string"),  # unseen category
    "has_elevator": pd.Series([pd.NA], dtype="boolean"),
})

X_test_demo = pd.DataFrame({
    "building_size": [110.0],
    "rooms_count": pd.Series([2], dtype="Int64"),
    "city_slug": pd.Series(["karaj"], dtype="string"),
    "has_elevator": pd.Series([False], dtype="boolean"),
})

preprocessing_summary(X_train_demo)


## Unscaled preprocessing

Use this version for models that do not require feature scaling, especially tree-based models.

In [ ]:
preprocessor_unscaled, groups = build_preprocessor(
    X_train_demo,
    scale_numeric=False,
)

groups


## Scaled preprocessing

For scale-sensitive algorithms, set `scale_numeric=True`. Scaling is applied **after numeric imputation** and its mean/standard deviation are learned from Train only.

In [ ]:
preprocessor_scaled, _ = build_preprocessor(
    X_train_demo,
    scale_numeric=True,
)

X_train_t, X_validation_t, X_test_t = fit_transform_feature_splits(
    preprocessor_scaled,
    X_train_demo,
    X_validation_demo,
    X_test_demo,
)

print("Train transformed shape:", X_train_t.shape)
print("Validation transformed shape:", X_validation_t.shape)
print("Test transformed shape:", X_test_t.shape)
print()
print("Output features:")
print(preprocessor_scaled.get_feature_names_out())


### Why this is leakage-safe

The helper performs exactly this sequence:

```python
X_train_t = preprocessor.fit_transform(X_train)
X_validation_t = preprocessor.transform(X_validation)
X_test_t = preprocessor.transform(X_test)
```

Therefore medians, category vocabularies, and scaling statistics are learned **only from Train**. Validation and Test never participate in fitting.

## Usage with the real Task-5 splits

Once Tasks 1–5 provide the final modeling features and split, the same code is used directly:

```python
from scripts.modeling_preprocessing import build_preprocessor, fit_transform_feature_splits

# Tree-based model preprocessing
preprocessor_tree, feature_groups = build_preprocessor(
    X_train,
    scale_numeric=False,
)

X_train_tree, X_validation_tree, X_test_tree = fit_transform_feature_splits(
    preprocessor_tree,
    X_train,
    X_validation,
    X_test,
)

# Scale-sensitive model preprocessing
preprocessor_scaled, _ = build_preprocessor(
    X_train,
    scale_numeric=True,
)
```

For Task 7, preprocessing and each estimator can also be combined in one sklearn `Pipeline` using `build_model_pipeline`, so model training remains leakage-safe end-to-end.